# 🎓 Few-Shot Benchmark on 22 Datasets across 8 Evaluated Models & Baselines
## Multimodal Vision-Language Few-Shot Adaptation (K = 1, 2, 4, 8, 16 Shots)

**Nghiên cứu & Thực nghiệm:**
- **Backbone VLM:** OpenAI CLIP ViT-B/32 (`openai/clip-vit-base-patch32`)
- **Backbone CNN:** ResNet-50 (`ResNet50_Weights.DEFAULT`)
- **Local SLM:** Google Gemma 4 (`google/gemma-4-e4b`) qua LM Studio trích xuất đặc trưng hình thái học

**8 Mô hình và Baselines được đánh giá:**
1. ⭐ `Dual-Stream Fusion` (Ours SOTA - Dung hợp Soft Prompt và Morphological Centroid Bank)
2. `Gemma-CoOp (Attribute)` (Context Optimization có điều kiện hóa bởi thuộc tính Gemma)
3. `CoOp (Baseline)` (Zhou et al., IJCV 2022)
4. `CLIP-Adapter` (Gao et al., IJCV 2024)
5. `Tip-Adapter` (Zhang et al., ECCV 2022)
6. `CLIP Baseline (Handcrafted)` (Zero-shot truyền thống)
7. `ResNet-50 Cosine Head (Ref)`
8. `ResNet-50 Prototype` (ProtoNet baseline)

**Quy mô Thực nghiệm:** 22 Datasets chuẩn quốc tế, 5 mốc K-shot (1, 2, 4, 8, 16), lưới đồ thị 4x6 gồm 24 panels chuẩn báo báo khoa học.

In [ ]:
# 1. Environment Setup & Hardware Diagnostics
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import Image, display
import torch
import torch.nn as nn
import torch.nn.functional as F

SEMESTER_DIR = os.path.abspath(os.getcwd())
RESULTS_KSHOT_DIR = os.path.join(SEMESTER_DIR, "results_kshot")
PLOTS_KSHOT_DIR = os.path.join(SEMESTER_DIR, "plots_kshot")
ATTRS_DIR = os.path.join(SEMESTER_DIR, "attributes")

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Active Compute Device: {device}")
print(f"📁 Working Directory: {SEMESTER_DIR}")
print(f"📊 Results directory exists: {os.path.exists(RESULTS_KSHOT_DIR)}")

In [ ]:
# 2. Model & Dataset Registries (Publication-Grade Styling)
MODEL_SPECS = [
    {
        "id": "dual_stream",
        "name": "Dual-Stream Fusion",
        "color": "#991B1B", "marker": "*", "ms": 11.5, "mfc": "#991B1B", "mec": "#991B1B",
        "lw": 2.2, "ls": "-", "mew": 1.2, "zorder": 10
    },
    {
        "id": "gemma_coop",
        "name": "Gemma-CoOp (Attribute)",
        "color": "#E11D48", "marker": "D", "ms": 5.5, "mfc": "#E11D48", "mec": "white",
        "lw": 1.7, "ls": "-", "mew": 0.8, "zorder": 9
    },
    {
        "id": "coop",
        "name": "CoOp (Baseline)",
        "color": "#D97706", "marker": "o", "ms": 5.5, "mfc": "#D97706", "mec": "white",
        "lw": 1.6, "ls": "-", "mew": 0.8, "zorder": 8
    },
    {
        "id": "clip_adapter",
        "name": "CLIP-Adapter",
        "color": "#7C3AED", "marker": "^", "ms": 6.0, "mfc": "#7C3AED", "mec": "white",
        "lw": 1.5, "ls": "-", "mew": 0.8, "zorder": 7
    },
    {
        "id": "tip_adapter",
        "name": "Tip-Adapter",
        "color": "#0284C7", "marker": "v", "ms": 6.0, "mfc": "#0284C7", "mec": "white",
        "lw": 1.5, "ls": "-", "mew": 0.8, "zorder": 6
    },
    {
        "id": "clip_baseline",
        "name": "CLIP Baseline (Handcrafted)",
        "color": "#64748B", "marker": "*", "ms": 9.5, "mfc": "white", "mec": "#64748B",
        "lw": 1.5, "ls": "--", "mew": 1.4, "zorder": 5
    },
    {
        "id": "resnet_cosine",
        "name": "ResNet-50 Cosine Head (Ref)",
        "color": "#854D0E", "marker": "+", "ms": 8.5, "mfc": "#854D0E", "mec": "#854D0E",
        "lw": 1.8, "ls": "-", "mew": 2.0, "zorder": 4
    },
    {
        "id": "resnet_proto",
        "name": "ResNet-50 Prototype",
        "color": "#15803D", "marker": "x", "ms": 7.0, "mfc": "#15803D", "mec": "#15803D",
        "lw": 1.5, "ls": ":", "mew": 1.8, "zorder": 3
    }
]

DATASETS_REGISTRY = [
    {"id": "oxford_pets", "name": "Oxford-IIIT Pet"},
    {"id": "vt_dataset", "name": "VtDataset (NewDataset)"},
    {"id": "caltech101", "name": "Caltech-101"},
    {"id": "cifar10", "name": "CIFAR-10"},
    {"id": "cifar100", "name": "CIFAR-100"},
    {"id": "dtd", "name": "Describable Textures (DTD)"},
    {"id": "eurosat", "name": "EuroSAT"},
    {"id": "flowers102", "name": "Oxford Flowers-102"},
    {"id": "food101", "name": "Food-101"},
    {"id": "gtsrb", "name": "GTSRB"},
    {"id": "mnist", "name": "MNIST"},
    {"id": "stl10", "name": "STL-10"},
    {"id": "sun397", "name": "SUN397"},
    {"id": "fgvc_aircraft", "name": "FGVC Aircraft"},
    {"id": "fer2013", "name": "FER2013"},
    {"id": "country211", "name": "Country211"},
    {"id": "clevr_counts", "name": "CLEVR Counts"},
    {"id": "resisc45", "name": "NWPU-RESISC45"},
    {"id": "voc2007", "name": "Pascal VOC 2007"},
    {"id": "pcam", "name": "PatchCamelyon (PCam)"},
    {"id": "rendered_sst2", "name": "Rendered SST2"},
    {"id": "ucf101", "name": "UCF101"}
]

SHOTS_LIST = [1, 2, 4, 8, 16]
print(f"✅ Configured {len(MODEL_SPECS)} models across {len(DATASETS_REGISTRY)} datasets.")

In [ ]:
# 3. PyTorch Architecture Reference Definitions

# (A) Residual Visual MLP Adapter
class CLIPAdapterModel(nn.Module):
    def __init__(self, text_features, bottleneck_dim=64, alpha=0.2, logit_scale=100.0):
        super().__init__()
        self.register_buffer("text_features", text_features.detach().clone())
        self.alpha = alpha
        self.logit_scale = logit_scale
        input_dim = text_features.shape[1]
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, bottleneck_dim, bias=False),
            nn.ReLU(),
            nn.Linear(bottleneck_dim, input_dim, bias=False)
        )
        nn.init.constant_(self.mlp[2].weight, 0.0)

    def forward(self, img_features):
        img_norm = F.normalize(img_features, dim=-1)
        res = self.mlp(img_norm)
        adapted = F.normalize((1.0 - self.alpha) * img_norm + self.alpha * res, dim=-1)
        return (adapted @ self.text_features.t()) * self.logit_scale

# (B) Key-Value Cache Tip-Adapter
class TipAdapterFModel(nn.Module):
    def __init__(self, train_keys, train_labels, text_features, alpha=1.0, beta=5.5, logit_scale=100.0):
        super().__init__()
        self.register_buffer("text_features", text_features.detach().clone())
        self.num_classes = text_features.shape[0]
        self.alpha = alpha
        self.beta = beta
        self.logit_scale = logit_scale
        cache_values = F.one_hot(train_labels, num_classes=self.num_classes).float()
        self.register_buffer("cache_values", cache_values)
        self.cache_keys = nn.Parameter(train_keys.clone())

    def forward(self, img_features):
        img_norm = F.normalize(img_features, dim=-1)
        keys_norm = F.normalize(self.cache_keys, dim=-1)
        sim_cache = img_norm @ keys_norm.t()
        affinity = torch.exp(-self.beta * (1.0 - sim_cache))
        logits_cache = affinity @ self.cache_values
        logits_clip = (img_norm @ self.text_features.t()) * self.logit_scale
        return logits_clip + self.alpha * logits_cache

# (C) Proposed Dual-Stream Fusion (Ours SOTA)
class DualStreamFusionModel(nn.Module):
    def __init__(self, W_centroid, bottleneck_dim=64, alpha=0.2, beta=0.35, logit_scale=100.0):
        super().__init__()
        self.register_buffer("W_centroid", W_centroid.detach().clone())
        self.alpha = alpha
        self.beta = beta
        self.logit_scale = logit_scale
        input_dim = W_centroid.shape[1]
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, bottleneck_dim, bias=False),
            nn.ReLU(),
            nn.Linear(bottleneck_dim, input_dim, bias=False)
        )
        nn.init.constant_(self.mlp[2].weight, 0.0)

    def forward(self, img_features, coop_text_features):
        img_norm = F.normalize(img_features, dim=-1)
        res = self.mlp(img_norm)
        adapted = F.normalize((1.0 - self.alpha) * img_norm + self.alpha * res, dim=-1)
        l_coop = (adapted @ coop_text_features.t()) * self.logit_scale
        l_cent = (adapted @ self.W_centroid.t()) * self.logit_scale
        return (1.0 - self.beta) * l_coop + self.beta * l_cent

print("✅ Neural architectures loaded and ready.")

In [ ]:
# 4. Inspect K-Shot Performance on Oxford-IIIT Pet & VtDataset
def display_dataset_summary(dataset_id):
    filepath = os.path.join(RESULTS_KSHOT_DIR, f"{dataset_id}.json")
    if not os.path.exists(filepath):
        print(f"File {filepath} not found.")
        return None
    with open(filepath, "r") as f:
        data = json.load(f)
    shots = data["shots"]
    curves = data["curves"]
    rows = []
    for spec in MODEL_SPECS:
        m_id = spec["id"]
        if m_id in curves:
            row = {"Model": spec["name"]}
            for s, acc in zip(shots, curves[m_id]):
                row[f"{s}-shot"] = f"{acc:.2f}%"
            rows.append(row)
    df = pd.DataFrame(rows)
    print(f"\n=== Dataset: {dataset_id.upper()} ===")
    return df

df_pets = display_dataset_summary("oxford_pets")
display(df_pets)

df_vt = display_dataset_summary("vt_dataset")
display(df_vt)

In [ ]:
# 5. Standalone 4x6 Publication-Grade Grid Plotter (24 Panels)
def render_and_show_grid():
    plt.rcParams['font.family'] = 'sans-serif'
    fig, axes = plt.subplots(6, 4, figsize=(21.0, 27.0), dpi=200)
    fig.patch.set_facecolor('white')
    axes = axes.flatten()
    
    shots_default = [1, 2, 4, 8, 16]
    all_dataset_curves = {spec["id"]: [] for spec in MODEL_SPECS}
    
    for idx, ds_meta in enumerate(DATASETS_REGISTRY):
        ax = axes[idx]
        ax.set_facecolor('white')
        ds_id = ds_meta["id"]
        ds_name = ds_meta["name"]
        
        res_file = os.path.join(RESULTS_KSHOT_DIR, f"{ds_id}.json")
        with open(res_file, "r") as f:
            res_data = json.load(f)
        x_vals = res_data["shots"]
        curves = res_data["curves"]
        
        for spec in MODEL_SPECS:
            all_dataset_curves[spec["id"]].append(curves[spec["id"]])
            
        for spec in MODEL_SPECS:
            m_id = spec["id"]
            y = np.array(curves[m_id])
            ax.plot(x_vals, y, color=spec["color"], linewidth=spec["lw"] * 1.05, linestyle=spec["ls"],
                    marker=spec["marker"], markerfacecolor=spec["mfc"], markeredgecolor=spec["mec"],
                    markeredgewidth=spec["mew"], markersize=spec["ms"] * 0.85, zorder=spec["zorder"])
                    
        ax.set_title(ds_name, fontsize=11, fontweight='bold', pad=7, color='#0F172A')
        ax.set_ylabel("Top-1 Accuracy (%)", fontsize=9.2, color='#1E293B', labelpad=3)
        ax.set_xlabel("Shots per Class (K)", fontsize=9.2, color='#1E293B', labelpad=3)
        ax.set_xticks(x_vals)
        ax.set_xticklabels([str(s) for s in x_vals])
        ax.tick_params(axis='both', which='major', labelsize=8.5)
        ax.grid(True, linestyle='--', alpha=0.55, color='#E2E8F0')
        
    # Panel 22: Overall Macro-Average across 22 Datasets
    ax_avg = axes[22]
    ax_avg.set_facecolor('#F8FAFC')
    for spec in MODEL_SPECS:
        m_id = spec["id"]
        arr = np.array(all_dataset_curves[m_id])
        mean_y = arr.mean(axis=0)
        ax_avg.plot(shots_default, mean_y, color=spec["color"], linewidth=spec["lw"] * 1.3, linestyle=spec["ls"],
                    marker=spec["marker"], markerfacecolor=spec["mfc"], markeredgecolor=spec["mec"],
                    markeredgewidth=spec["mew"] * 1.1, markersize=spec["ms"] * 0.95, zorder=spec["zorder"])
                    
    ax_avg.set_title("★ 22-Dataset Macro Average", fontsize=11.5, fontweight='bold', pad=7, color='#B91C1C')
    ax_avg.set_ylabel("Mean Top-1 Accuracy (%)", fontsize=9.2, color='#1E293B', labelpad=3)
    ax_avg.set_xlabel("Shots per Class (K)", fontsize=9.2, color='#1E293B', labelpad=3)
    ax_avg.set_xticks(shots_default)
    ax_avg.set_xticklabels([str(s) for s in shots_default])
    ax_avg.tick_params(axis='both', which='major', labelsize=8.5)
    ax_avg.grid(True, linestyle='--', alpha=0.6, color='#CBD5E1')
    
    # Panel 23: Publication Legend
    ax_leg = axes[23]
    ax_leg.set_facecolor('white')
    ax_leg.axis('off')
    
    legend_elements = [
        Line2D([0], [0], color=spec["color"], lw=spec["lw"] * 1.2, linestyle=spec["ls"],
               marker=spec["marker"], markersize=spec["ms"] * 0.95,
               markerfacecolor=spec["mfc"], markeredgecolor=spec["mec"],
               markeredgewidth=spec["mew"], label=spec["name"]) for spec in MODEL_SPECS
    ]
    leg = ax_leg.legend(
        handles=legend_elements, loc='center', title="Evaluated Models & Baselines",
        title_fontsize=12.5, fontsize=10.5, frameon=True, framealpha=0.98,
        facecolor='#F8FAFC', edgecolor='#CBD5E1', labelspacing=0.9, handlelength=2.8, borderpad=1.4
    )
    leg.get_frame().set_boxstyle('round,pad=0.9')
    leg.get_title().set_fontweight('bold')
    leg.get_title().set_color('#0F172A')
    
    plt.tight_layout(pad=2.8)
    out_grid = os.path.join(PLOTS_KSHOT_DIR, "benchmark_22_datasets_kshot_4x6_grid.png")
    fig.savefig(out_grid, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved & Displayed grid image: {out_grid}")

render_and_show_grid()